# Six-run training comparison

Compares RoBERTa full fine-tuning and Qwen LoRA across three variants:

- Static weighting with linear heads.
- Adaptive minimum-variety weighting with linear heads.
- Adaptive minimum-variety weighting with MLP heads.

Each result contains split seeds 2026, 2027, and 2028. The benchmark score is

$$
\frac{\min(F1_{sent,AU},F1_{sent,UK})+\min(F1_{sarc,AU},F1_{sarc,UK})}{2}.
$$


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
})

def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "training_summary" / "runs").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate training_summary/runs")

REPO_ROOT = find_repo_root()
RUN_ROOT = REPO_ROOT / "training_summary" / "runs"
SEEDS = (2026, 2027, 2028)

RUNS = {
    "RoBERTa · Static": {
        "path": "roberta_multitask", "model": "RoBERTa", "variant": "Static"
    },
    "RoBERTa · Adaptive linear": {
        "path": "roberta_adaptive_minvariety", "model": "RoBERTa", "variant": "Adaptive linear"
    },
    "RoBERTa · Adaptive MLP": {
        "path": "roberta_adaptive_minvariety_mlp", "model": "RoBERTa", "variant": "Adaptive MLP"
    },
    "Qwen · Static": {
        "path": "qwen_embedding_0.6b_multitask_lora", "model": "Qwen", "variant": "Static"
    },
    "Qwen · Adaptive linear": {
        "path": "qwen_embedding_0.6b_lora_adaptive_minvariety", "model": "Qwen", "variant": "Adaptive linear"
    },
    "Qwen · Adaptive MLP": {
        "path": "qwen_embedding_0.6b_lora_adaptive_minvariety_mlp", "model": "Qwen", "variant": "Adaptive MLP"
    },
}

VARIANTS = ("Static", "Adaptive linear", "Adaptive MLP")
MODELS = ("RoBERTa", "Qwen")
COLORS = {
    "Static": "#4C78A8",
    "Adaptive linear": "#E45756",
    "Adaptive MLP": "#72B7B2",
}

missing = [
    str(RUN_ROOT / spec["path"] / "all_results.json")
    for spec in RUNS.values()
    if not (RUN_ROOT / spec["path"] / "all_results.json").is_file()
]
if missing:
    raise FileNotFoundError(f"Missing run summaries: {missing}")

print(f"Loaded configuration for {len(RUNS)} runs from {RUN_ROOT}")


In [ ]:
SUBTASKS = {
    "Sentiment AU": "en_AU_sentiment_macro_f1",
    "Sentiment UK": "en_UK_sentiment_macro_f1",
    "Sarcasm AU": "en_AU_sarcasm_macro_f1",
    "Sarcasm UK": "en_UK_sarcasm_macro_f1",
}

def benchmark_score(metrics):
    return (
        min(metrics["en_AU_sentiment_macro_f1"], metrics["en_UK_sentiment_macro_f1"])
        + min(metrics["en_AU_sarcasm_macro_f1"], metrics["en_UK_sarcasm_macro_f1"])
    ) / 2

history_rows, test_rows, config_rows = [], [], []
for run, spec in RUNS.items():
    root = RUN_ROOT / spec["path"]
    first_metadata = json.loads((root / "seed_2026" / "metadata.json").read_text())
    cfg = first_metadata["training_config"]
    config_rows.append({
        "run": run,
        "epochs": cfg["epochs"],
        "effective_batch": cfg["train_batch_size"] * cfg["gradient_accumulation_steps"],
        "learning_rate": cfg["learning_rate"],
        "weight_decay": cfg["weight_decay"],
        "warmup_ratio": cfg["warmup_ratio"],
        "max_length": cfg["max_length"],
        "dropout": cfg["dropout"],
        "precision": cfg["mixed_precision"],
        "loss_weighting": cfg.get("loss_weighting", "task_class"),
        "selection": cfg.get("selection_metric", "overall_mean_macro_f1"),
        "head": cfg.get("head_type", "linear"),
        "trainable_M": first_metadata["parameter_counts"]["trainable"] / 1e6,
    })

    for seed in SEEDS:
        epoch_history = json.loads((root / f"seed_{seed}" / "history.json").read_text())
        result = json.loads((root / f"seed_{seed}" / "test_metrics.json").read_text())
        for epoch_result in epoch_history:
            history_rows.append({
                "run": run, "model": spec["model"], "variant": spec["variant"],
                "seed": seed, "benchmark_score": benchmark_score(epoch_result),
                **epoch_result,
            })
        metrics = result["test_metrics"]
        test_rows.append({
            "run": run, "model": spec["model"], "variant": spec["variant"],
            "seed": seed, "selected_epoch": result["selected_epoch"],
            "benchmark_score": benchmark_score(metrics), **metrics,
        })

configs = pd.DataFrame(config_rows).set_index("run")
history = pd.DataFrame(history_rows)
test = pd.DataFrame(test_rows)

display(configs)


### Comparability

- Equal: five epochs, effective batch 16, splits, seeds, max length, dropout, weight decay, warmup, and BF16 configuration.
- Model-specific: RoBERTa full fine-tuning at `2e-5`; Qwen LoRA at `2e-4`.
- Intentional variant changes: head architecture, loss weighting, and checkpoint-selection metric.
- Therefore, static versus adaptive is a system comparison, not an isolated weighting ablation.


In [ ]:
summary_rows = []
for run, part in test.groupby("run", sort=False):
    row = {"Run": run}
    for label, metric in SUBTASKS.items():
        row[label] = f"{part[metric].mean():.4f} ± {part[metric].std():.4f}"
    row["Score"] = f"{part.benchmark_score.mean():.4f} ± {part.benchmark_score.std():.4f}"
    row["Best epochs"] = ", ".join(map(str, part.sort_values("seed").selected_epoch))
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows).set_index("Run")
display(summary)


## Benchmark score


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(MODELS))
width = 0.24
offsets = np.linspace(-width, width, len(VARIANTS))

for offset, variant in zip(offsets, VARIANTS):
    means, stds = [], []
    for model in MODELS:
        values = test[(test.model == model) & (test.variant == variant)].benchmark_score
        means.append(values.mean())
        stds.append(values.std())
    positions = x + offset
    ax.bar(positions, means, width, yerr=stds, capsize=4,
           color=COLORS[variant], alpha=0.78, label=variant)
    for j, model in enumerate(MODELS):
        values = test[(test.model == model) & (test.variant == variant)].benchmark_score.to_numpy()
        ax.scatter(np.full(len(values), positions[j]) + np.linspace(-0.025, 0.025, len(values)),
                   values, color="black", s=22, zorder=3)

ax.set_xticks(x, MODELS)
ax.set_ylabel("Untouched-test benchmark score")
ax.set_ylim(0.74, 0.84)
ax.set_title("Mean ± sample SD; dots are split seeds")
ax.legend(frameon=False, ncol=3)
fig.tight_layout()
plt.show()


## Four benchmark components


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8), sharex=True)
x = np.arange(len(MODELS))
offsets = np.linspace(-0.22, 0.22, len(VARIANTS))

for ax, (label, metric) in zip(axes.flat, SUBTASKS.items()):
    for offset, variant in zip(offsets, VARIANTS):
        for model_index, model in enumerate(MODELS):
            values = test[(test.model == model) & (test.variant == variant)][metric]
            position = x[model_index] + offset
            ax.errorbar(position, values.mean(), yerr=values.std(), fmt="o",
                        capsize=4, color=COLORS[variant], markersize=7)
            ax.scatter(np.full(len(values), position) + np.linspace(-0.018, 0.018, len(values)),
                       values, color="black", s=13, zorder=3)
    ax.set_title(label)
    ax.set_xticks(x, MODELS)
    ax.set_ylabel("Test macro-F1")

handles = [plt.Line2D([0], [0], marker="o", linestyle="", color=COLORS[v], label=v) for v in VARIANTS]
fig.legend(handles=handles, loc="upper center", ncol=3, frameon=False)
fig.suptitle("Task–variety performance", y=0.98, fontsize=14)
fig.tight_layout(rect=(0, 0, 1, 0.94))
plt.show()


## Paired change relative to each model's static run


In [ ]:
delta_rows = []
for model in MODELS:
    baseline = test[(test.model == model) & (test.variant == "Static")].set_index("seed")
    for variant in ("Adaptive linear", "Adaptive MLP"):
        candidate = test[(test.model == model) & (test.variant == variant)].set_index("seed")
        deltas = candidate.benchmark_score - baseline.benchmark_score
        delta_rows.append({
            "comparison": f"{model} · {variant}",
            **{str(seed): deltas.loc[seed] for seed in SEEDS},
            "Mean": deltas.mean(),
        })

deltas = pd.DataFrame(delta_rows).set_index("comparison")
fig, ax = plt.subplots(figsize=(9, 4))
limit = np.abs(deltas.to_numpy()).max()
image = ax.imshow(deltas, cmap="RdBu", vmin=-limit, vmax=limit, aspect="auto")
ax.set_xticks(range(len(deltas.columns)), deltas.columns)
ax.set_yticks(range(len(deltas.index)), deltas.index)
ax.set_title("Adaptive − static benchmark score")
for row in range(deltas.shape[0]):
    for column in range(deltas.shape[1]):
        value = deltas.iloc[row, column]
        ax.text(column, row, f"{value:+.3f}", ha="center", va="center",
                color="white" if abs(value) > limit * 0.55 else "black")
fig.colorbar(image, ax=ax, label="Score change", shrink=0.85)
fig.tight_layout()
plt.show()


## Validation trajectories


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharex=True, sharey=True)
for ax, run in zip(axes.flat, RUNS):
    for seed in SEEDS:
        part = history[(history.run == run) & (history.seed == seed)]
        ax.plot(part.epoch, part.benchmark_score, marker="o", linewidth=1.6, label=str(seed))
        selected_epoch = int(test[(test.run == run) & (test.seed == seed)].selected_epoch.iloc[0])
        selected_score = part.loc[part.epoch == selected_epoch, "benchmark_score"].iloc[0]
        ax.scatter(selected_epoch, selected_score, marker="*", s=90,
                   edgecolor="black", linewidth=0.5, zorder=4)
    ax.set_title(run)
    ax.set_xticks(range(1, 6))
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Validation benchmark score")
axes[0, -1].legend(title="Seed", frameon=False)
fig.suptitle("Stars mark the checkpoint selected by each run's configured metric", y=1.01, fontsize=14)
fig.tight_layout()
plt.show()


## Adaptive loss weights


In [ ]:
adaptive_runs = [run for run, spec in RUNS.items() if spec["variant"] != "Static"]
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey=True)

for ax, run in zip(axes.flat, adaptive_runs):
    part = history[history.run == run]
    for seed in SEEDS:
        seed_part = part[part.seed == seed]
        sentiment_au = [w["sentiment"]["en_AU"] for w in seed_part.training_variety_weights]
        sarcasm_uk = [w["sarcasm"]["en_UK"] for w in seed_part.training_variety_weights]
        color = f"C{SEEDS.index(seed)}"
        ax.plot(seed_part.epoch, sentiment_au, color=color, marker="o", linewidth=1.5)
        ax.plot(seed_part.epoch, sarcasm_uk, color=color, marker="s", linestyle="--", linewidth=1.5)
    ax.axhline(0.5, color="grey", linewidth=1)
    ax.set_title(run)
    ax.set_xlabel("Epoch using weight")
    ax.set_ylabel("Share of task loss")
    ax.set_xticks(range(1, 6))

seed_handles = [plt.Line2D([0], [0], color=f"C{i}", label=str(seed)) for i, seed in enumerate(SEEDS)]
target_handles = [
    plt.Line2D([0], [0], color="black", marker="o", label="Sentiment AU"),
    plt.Line2D([0], [0], color="black", marker="s", linestyle="--", label="Sarcasm UK"),
]
fig.legend(handles=seed_handles + target_handles, loc="upper center", ncol=5, frameon=False)
fig.suptitle("Weight assigned to the recurring weaker variety", y=0.99, fontsize=14)
fig.tight_layout(rect=(0, 0, 1, 0.94))
plt.show()


## Concise findings


In [ ]:
ranked = test.groupby("run").benchmark_score.agg(["mean", "std"]).sort_values("mean", ascending=False)
best_run = ranked.index[0]
roberta_static = ranked.loc["RoBERTa · Static", "mean"]
roberta_mlp = ranked.loc["RoBERTa · Adaptive MLP", "mean"]
qwen_static = ranked.loc["Qwen · Static", "mean"]
qwen_linear = ranked.loc["Qwen · Adaptive linear", "mean"]
qwen_mlp = ranked.loc["Qwen · Adaptive MLP", "mean"]

display(Markdown(f'''
- Best mean: **{best_run}**, `{ranked.iloc[0]['mean']:.4f} ± {ranked.iloc[0]['std']:.4f}`.
- Qwen adaptive linear and MLP are effectively tied: `{qwen_linear:.4f}` versus `{qwen_mlp:.4f}`.
- RoBERTa adaptive MLP improves over static by `{roberta_mlp - roberta_static:+.4f}`; adaptive linear improves by `{ranked.loc['RoBERTa · Adaptive linear', 'mean'] - roberta_static:+.4f}`.
- Qwen adaptive gains over static are seed-sensitive; compare paired deltas, not only means.
- en_AU sentiment and en_UK sarcasm are the minima in every run; en_UK sarcasm remains the main bottleneck.
- Static and adaptive use different objectives and selection metrics. Add a non-adaptive variety-balanced run with minimum-score selection for a clean weighting ablation.
'''))
